## 📊 Project Description

This project analyzes user reviews of ChatGPT to uncover:

- Sentiment trends (positive, negative, neutral)
- Common user praises and complaints
- Product weaknesses and strengths
- Evolution of user perception over time

In [1]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from collections import Counter
from textblob import TextBlob
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.templates.default = 'plotly_white'

### Loading the data

In [2]:
df = pd.read_csv(r"C:\Users\HP\Documents\Data journey\Chatgpt\src\data\chatgpt_reviews.csv")
df.head()

,Review Id,Review,Ratings,Review Date
0,6fb93778-651a-4ad1-b5ed-67dd0bd35aac,good,5,2024-08-23 19:30:05
1,81caeefd-3a28-4601-a898-72897ac906f5,good,5,2024-08-23 19:28:18
2,452af49e-1d8b-4b68-b1ac-a94c64cb1dd5,nice app,5,2024-08-23 19:22:59
3,372a4096-ee6a-4b94-b046-cef0b646c965,"nice, ig",5,2024-08-23 19:20:50
4,b0d66a4b-9bde-4b7c-8b11-66ed6ccdd7da,"this is a great app, the bot is so accurate to...",5,2024-08-23 19:20:39


### Pre processing

In [3]:
df.isna().sum()

Review Id      0
Review         6
Ratings        0
Review Date    0
dtype: int64

In [4]:
df['Review'] = df['Review'].fillna('NO REVIEWS')

### Sentiment Analysis

In [5]:
#function to determine sentiment polarity
def get_sentiment(review):
    sentiment = TextBlob(review).sentiment.polarity
    if sentiment > 0:
      return 'Positive'
    elif sentiment < 0:
      return 'Negative'
    else:
      return 'Neutral'

In [6]:
#applying the function
df['Sentiment'] = df['Review'].apply(get_sentiment)

In [7]:
sentiments = df['Sentiment'].value_counts()

### Visualizing

In [8]:
output_dir = r"C:\Users\HP\Documents\Data journey\Chatgpt\outputs"

In [9]:
fig = go.Figure(data = [go.Bar(
    x = sentiments.index,
    y = sentiments.values,
    marker_color = ['green', 'grey', 'red']
)])
fig.update_layout(
    title = 'Sentiment Distribution of Reviews',
    xaxis_title = 'Sentiment',
    yaxis_title = 'Count'
)

fig.write_image(f'{output_dir}\Sentiment Distribution of Reviews.png')
fig.show()

### Text Mining

In [10]:
#first we'll filter out the positive reviews
positive_reviews = df[df['Sentiment'] == 'Positive']

In [11]:
#use vectorizer to extract common phrases
vectorizer = CountVectorizer(stop_words = 'english', ngram_range = (2,3), max_features = 100)
X = vectorizer.fit_transform(positive_reviews['Review'])

In [12]:
#sum the counts of each phrase
phrase_counts = X.sum(axis = 0)
phrases = vectorizer.get_feature_names_out()
phrase_frequency = [(phrases[i], phrase_counts[0,i]) for i in range(len(phrases))]

#sort phrases by frequency
phrase_frequency = sorted(phrase_frequency, key = lambda x:x[1], reverse = True)

#turning it to a dataframe
phrase_df = pd.DataFrame(phrase_frequency, columns = ['Phrase', 'Frequency'])

In [13]:
negative_reviews = df[df['Sentiment'] == 'Negative']

In [14]:
X_neg = vectorizer.fit_transform(negative_reviews['Review'])

In [15]:
#sum the counts of each phrase
negative_phrase_counts = X_neg.sum(axis = 0)
negative_phrases = vectorizer.get_feature_names_out()
negative_phrase_frequency = [(negative_phrases[i], negative_phrase_counts[0,i]) for i in range(len(negative_phrases))]

#sort phrases by frequency
negative_phrase_frequency = sorted(negative_phrase_frequency, key = lambda x:x[1], reverse = True)

#turning it to a dataframe
negative_phrase_df = pd.DataFrame(negative_phrase_frequency, columns = ['Phrase', 'Frequency'])

### Visualizing

In [16]:
#visualize
fig = px.bar(
    phrase_df,
    x = 'Frequency',
    y = 'Phrase',
    orientation = 'h',
    title = 'Common Phrases in Positive Reviews',
    labels = {'Phrase': 'Phrase', 'Frequency': 'Frequency'},
    color_discrete_sequence = ['green'],
    width = 1000,
    height = 600
)

fig.update_layout(
    xaxis_title = 'Phrase',
    yaxis_title = 'Frequency',
    yaxis  ={'categoryorder': 'total ascending'}
)

fig.write_image(f'{output_dir}\Common Phrases in Positive Reviews.png')
fig.show()

In [17]:
#visualize
fig = px.bar(
    negative_phrase_df,
    x = 'Frequency',
    y = 'Phrase',
    orientation = 'h',
    title = 'Common Phrases in Negative Reviews',
    labels = {'Phrase': 'Negative Phrase', 'Frequency': 'Frequency'},
    color_discrete_sequence = ['green'],
    width = 1000,
    height = 600
)

fig.update_layout(
    xaxis_title = 'Negative Phrase',
    yaxis_title = 'Frequency',
    yaxis  ={'categoryorder': 'total ascending'}
)

fig.write_image(f'{output_dir}\Common Phrases in Negative Reviews.png')
fig.show()

### Common Chatgpt Problems

We want to group the common problems into categories such as
1. Quality of responses and answers
2. App performance
3. User interface
4. General features

In [18]:
#we'll first group the phrases into the categories
problem_categories = {
    'Responses and Answers Quality': ['wrong answer', 'gives wrong', 'incorrect', 'inaccurate', 'wrong', 'bad response',
                                      'irrelevant', 'useless', 'poor'],
    'App Performance': ['bad', 'lag', 'freeze', 'crash', 'bug', 'loading', 'glitch'],
    'User Interface': ['poor', 'interface', 'UI', 'layout', 'difficult', 'confusing'],
    'General Features': ['network', 'feature missing', 'poor', 'not working', 'not available', 'poor network', 'no network']
}

#a dictionary to count the occurence of the problem categories
problem_counts = {key: 0 for key in problem_categories.keys()}

In [19]:
#let's count
for phrase, count in negative_phrase_frequency:
  for category, keywords in problem_categories.items():
    if any(keyword in phrase for keyword in keywords):
      problem_counts[category] += count
      break

In [20]:
problem_df = pd.DataFrame(list(problem_counts.items()), columns = ['Problem Category', 'Count'])
problem_df.head()

,Problem Category,Count
0,Responses and Answers Quality,759
1,App Performance,219
2,User Interface,0
3,General Features,35


### Visualize

In [21]:
#visualize
fig = px.bar(
    problem_df,
    x = 'Problem Category',
    y = 'Count',
    title = 'Common Problems Encountered by Chatgpt',
    labels = {'Problem Category': 'Problem Category', 'Count': 'Frequency'}
)

fig.update_layout(
    xaxis_title = 'Problem Category',
    yaxis_title = 'Frequency',
    yaxis = {'categoryorder': 'total descending'}
)

fig.write_image(f'{output_dir}\Common Problems Encountered by Chatgpt.png')
fig.show()

### Review Trend Over Time

In [22]:
#convert the review date to datetime
df['Review Date'] = pd.to_datetime(df['Review Date'])

In [23]:
#aggregate sentiment counts by date
sentiment_overtime = df.groupby([df['Review Date'].dt.to_period('M'), 'Sentiment']).size().unstack(fill_value = 0)

#convert period back to datetime
sentiment_overtime.index = sentiment_overtime.index.to_timestamp()

### Visualize

In [24]:
#visualize
fig = go.Figure()

for sentiment in sentiment_overtime.columns:
  fig.add_trace(go.Scatter(
      x = sentiment_overtime.index,
      y = sentiment_overtime[sentiment],
      mode = 'lines',
      name = sentiment
  ))

  fig.update_layout(
      title = 'Sentiment Over Time',
      xaxis_title = 'Date',
      yaxis_title = 'Number of eviews',
      legend_title = 'Sentiment',
      xaxis=dict(showgrid=True, gridcolor='lightgray'),
      yaxis=dict(showgrid=True, gridcolor='lightgray')
  )

fig.write_image(f'{output_dir}\Sentiment Over Time.png')
fig.show()

### 📈 Insights & Analysis

#### Key Findings:

- Sentiment Distribution:
    Majority positive, but notable negative segment
- Top Positive Themes:
  “very helpful”
  “easy to use”
  “saves time”
- Top Complaints:
  Incorrect responses
- App performance issues (lag/crash):
  Missing features
- Trend Over Time:
  Increasing negative sentiment correlates with scaling issues